# 🫀 ML Assignment 2: Heart Disease Classification

**Course:** Machine Learning — M.Tech (AIML/DSE), BITS Pilani WILP

---

## 📋 Problem Statement
Predict the presence or absence of heart disease in a patient based on clinical and diagnostic features using multiple supervised classification models.

## 📊 Dataset
- **Name:** Heart Disease Dataset (UCI / Kaggle)
- **Source:** [Kaggle — Heart Disease Dataset](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset)
- **Instances:** 1,025
- **Features:** 13 predictive features + 1 target variable
- **Task:** Binary Classification (Heart Disease: Yes/No)

### Feature Descriptions
| # | Feature | Type | Description |
|---|---------|------|-------------|
| 1 | `age` | Numeric | Age of the patient in years |
| 2 | `sex` | Categorical | 1 = Male, 0 = Female |
| 3 | `cp` | Categorical | Chest pain type (0-3) |
| 4 | `trestbps` | Numeric | Resting blood pressure (mm Hg) |
| 5 | `chol` | Numeric | Serum cholesterol (mg/dl) |
| 6 | `fbs` | Categorical | Fasting blood sugar > 120 mg/dl (1=true, 0=false) |
| 7 | `restecg` | Categorical | Resting ECG results (0, 1, 2) |
| 8 | `thalach` | Numeric | Maximum heart rate achieved |
| 9 | `exang` | Categorical | Exercise-induced angina (1=yes, 0=no) |
| 10 | `oldpeak` | Numeric | ST depression induced by exercise |
| 11 | `slope` | Categorical | Slope of peak exercise ST segment |
| 12 | `ca` | Numeric | Number of major vessels colored by fluoroscopy (0-3) |
| 13 | `thal` | Categorical | Thalassemia (0=normal, 1=fixed defect, 2=reversible defect) |
| 14 | `target` | Label | 1 = Heart Disease, 0 = No Heart Disease |

---
## Step 0: Install Dependencies & Setup

In [ ]:
# Install required packages (Colab usually has these, but just in case)
!pip install -q scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report,
    RocCurveDisplay
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

# Serialization
import joblib
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

print('✅ All imports successful!')

---
## Step 1: Load the Dataset

### Option A: Download from Kaggle (Recommended)
1. Go to [Kaggle Heart Disease Dataset](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset)
2. Download `heart.csv`
3. Upload it to Colab using the file upload cell below

### Option B: Upload manually

In [ ]:
# ============================================================
# OPTION A: Upload heart.csv from your local machine
# ============================================================
from google.colab import files

print("📂 Please upload your 'heart.csv' file:")
uploaded = files.upload()

# The uploaded file name
filename = list(uploaded.keys())[0]
print(f"\n✅ File '{filename}' uploaded successfully!")

In [ ]:
# Load the dataset
df = pd.read_csv(filename)

print(f"📊 Dataset Shape: {df.shape}")
print(f"   Rows (instances): {df.shape[0]}")
print(f"   Columns (features + target): {df.shape[1]}")
print(f"\n✅ Minimum Feature Size (12): {'PASS ✓' if df.shape[1] - 1 >= 12 else 'FAIL ✗'}")
print(f"✅ Minimum Instance Size (500): {'PASS ✓' if df.shape[0] >= 500 else 'FAIL ✗'}")

---
## Step 2: Exploratory Data Analysis (EDA)

In [ ]:
# First 5 rows
print("=" * 60)
print("FIRST 5 ROWS")
print("=" * 60)
df.head()

In [ ]:
# Dataset info
print("=" * 60)
print("DATASET INFO")
print("=" * 60)
df.info()

In [ ]:
# Statistical summary
print("=" * 60)
print("STATISTICAL SUMMARY")
print("=" * 60)
df.describe().T

In [ ]:
# Check for missing values
print("=" * 60)
print("MISSING VALUES")
print("=" * 60)
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

if duplicates > 0:
    print(f"Removing {duplicates} duplicate rows...")
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Dataset shape after removing duplicates: {df.shape}")

In [ ]:
# Target distribution
print("=" * 60)
print("TARGET DISTRIBUTION")
print("=" * 60)

target_counts = df['target'].value_counts()
print(target_counts)
print(f"\nClass balance: {target_counts[0]/len(df)*100:.1f}% (No Disease) vs {target_counts[1]/len(df)*100:.1f}% (Disease)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
target_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Target Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Target (0 = No Disease, 1 = Disease)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No Disease (0)', 'Disease (1)'], rotation=0)

# Pie chart
axes[1].pie(target_counts, labels=['No Disease', 'Disease'], autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.02, 0.02),
            textprops={'fontsize': 12})
axes[1].set_title('Target Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, square=True,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature distributions
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    if i < len(axes):
        if df[col].nunique() <= 6:
            df[col].value_counts().sort_index().plot(kind='bar', ax=axes[i], color='#3498db',
                                                      edgecolor='black', alpha=0.8)
        else:
            axes[i].hist(df[col], bins=25, color='#3498db', edgecolor='black', alpha=0.8)
        axes[i].set_title(col, fontsize=12, fontweight='bold')
        axes[i].tick_params(labelsize=9)

# Hide unused subplots
for j in range(len(df.columns), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 3: Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"\nFeature columns ({X.shape[1]}): {list(X.columns)}")

In [ ]:
# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTrain target distribution:\n{y_train.value_counts()}")
print(f"\nTest target distribution:\n{y_test.value_counts()}")

In [ ]:
# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Feature scaling applied (StandardScaler)")
print(f"   Training mean ≈ {X_train_scaled.mean(axis=0).mean():.4f} (should be ~0)")
print(f"   Training std  ≈ {X_train_scaled.std(axis=0).mean():.4f} (should be ~1)")

In [ ]:
# Save the test data as CSV for Streamlit app upload
test_df = pd.DataFrame(X_test, columns=X.columns)
test_df['target'] = y_test.values
test_df.to_csv('test_data.csv', index=False)

print(f"✅ Test data saved to 'test_data.csv' ({test_df.shape[0]} rows, {test_df.shape[1]} columns)")
test_df.head()

---
## Step 4: Model Training & Evaluation

We will train the following 5 classification models:
1. **Logistic Regression**
2. **Decision Tree Classifier**
3. **K-Nearest Neighbor (KNN) Classifier**
4. **Gaussian Naive Bayes**
5. **Random Forest (Ensemble)**

In [ ]:
# Define all models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

print(f"📋 Models to train: {len(models)}")
for name in models:
    print(f"   • {name}")

In [ ]:
def evaluate_model(model, X_test, y_test):
    """
    Evaluate a trained model and return all required metrics.
    """
    y_pred = model.predict(X_test)

    # For AUC, we need probability estimates
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    else:
        auc = roc_auc_score(y_test, y_pred)

    metrics = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': auc,
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_test, y_pred)
    }
    return metrics, y_pred

In [ ]:
# Train all models and collect results
results = {}
predictions = {}
trained_models = {}

print("=" * 70)
print("MODEL TRAINING & EVALUATION")
print("=" * 70)

for name, model in models.items():
    print(f"\n🔄 Training: {name}...")

    # Train
    model.fit(X_train_scaled, y_train)

    # Evaluate
    metrics, y_pred = evaluate_model(model, X_test_scaled, y_test)

    results[name] = metrics
    predictions[name] = y_pred
    trained_models[name] = model

    print(f"   ✅ Accuracy: {metrics['Accuracy']:.4f} | AUC: {metrics['AUC']:.4f} | "
          f"F1: {metrics['F1']:.4f} | MCC: {metrics['MCC']:.4f}")

print("\n" + "=" * 70)
print("✅ ALL MODELS TRAINED SUCCESSFULLY!")
print("=" * 70)

---
## Step 5: Comparison Table (All 6 Metrics × 5 Models)

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.round(4)
comparison_df.index.name = 'ML Model Name'

print("=" * 80)
print("📊 MODEL COMPARISON TABLE")
print("=" * 80)
print()
comparison_df

In [ ]:
# Styled table with color highlighting (best values in green)
def highlight_best(s):
    """Highlight the maximum value in each column."""
    is_best = s == s.max()
    return ['background-color: #2ecc71; color: white; font-weight: bold' if v else '' for v in is_best]

styled_table = comparison_df.style.apply(highlight_best, axis=0).format('{:.4f}')
styled_table

In [ ]:
# Determine the overall winner
# Using F1 Score as the primary metric (harmonic mean of precision and recall)
winner = comparison_df['F1'].idxmax()
winner_f1 = comparison_df.loc[winner, 'F1']

print(f"\n🏆 OVERALL WINNER: {winner}")
print(f"   F1 Score: {winner_f1:.4f}")
print(f"   Accuracy: {comparison_df.loc[winner, 'Accuracy']:.4f}")
print(f"   AUC: {comparison_df.loc[winner, 'AUC']:.4f}")

In [ ]:
# Visual comparison - Bar chart of all metrics
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics_list = ['Accuracy', 'AUC', 'Precision', 'Recall', 'F1', 'MCC']
colors_bar = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

for idx, metric in enumerate(metrics_list):
    ax = axes[idx // 3][idx % 3]
    values = comparison_df[metric]
    bars = ax.bar(range(len(values)), values, color=colors_bar, edgecolor='black', alpha=0.85)

    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(values)))
    ax.set_xticklabels(['LR', 'DT', 'KNN', 'NB', 'RF'], fontsize=10)
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Model Performance Comparison (All Metrics)', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 6: Confusion Matrices

In [ ]:
# Plot confusion matrices for all models
fig, axes = plt.subplots(1, 5, figsize=(25, 4.5))
model_names = list(models.keys())

for idx, name in enumerate(model_names):
    cm = confusion_matrix(y_test, predictions[name])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'],
                cbar=False, linewidths=1, linecolor='black')
    axes[idx].set_title(name, fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual' if idx == 0 else '')

plt.suptitle('Confusion Matrices — All Models', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 7: ROC Curves

In [ ]:
# Plot ROC curves for all models
plt.figure(figsize=(10, 7))
colors_roc = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

for idx, (name, model) in enumerate(trained_models.items()):
    RocCurveDisplay.from_estimator(
        model, X_test_scaled, y_test,
        name=name, ax=plt.gca(), color=colors_roc[idx], linewidth=2
    )

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.title('ROC Curves — All Models', fontsize=16, fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8: Classification Reports (Detailed)

In [ ]:
# Print detailed classification reports
for name in model_names:
    print("=" * 60)
    print(f"Classification Report: {name}")
    print("=" * 60)
    print(classification_report(y_test, predictions[name],
                                target_names=['No Disease', 'Disease']))
    print()

---
## Step 9: Model Observations

The observations below will be printed and should be used for the README.md.

In [ ]:
# Generate observations based on actual results
print("=" * 80)
print("MODEL OBSERVATIONS")
print("=" * 80)

observations = {}

for name in model_names:
    m = results[name]
    obs_parts = []

    # Accuracy assessment
    if m['Accuracy'] >= 0.90:
        obs_parts.append(f"Achieved excellent accuracy of {m['Accuracy']:.2%}")
    elif m['Accuracy'] >= 0.80:
        obs_parts.append(f"Achieved good accuracy of {m['Accuracy']:.2%}")
    else:
        obs_parts.append(f"Achieved moderate accuracy of {m['Accuracy']:.2%}")

    # Precision vs Recall balance
    if abs(m['Precision'] - m['Recall']) < 0.05:
        obs_parts.append("Shows well-balanced precision and recall")
    elif m['Precision'] > m['Recall']:
        obs_parts.append(f"Higher precision ({m['Precision']:.2%}) than recall ({m['Recall']:.2%}), suggesting fewer false positives")
    else:
        obs_parts.append(f"Higher recall ({m['Recall']:.2%}) than precision ({m['Precision']:.2%}), better at capturing positive cases")

    # AUC assessment
    if m['AUC'] >= 0.90:
        obs_parts.append(f"Excellent AUC ({m['AUC']:.4f}), strong class separation")
    elif m['AUC'] >= 0.80:
        obs_parts.append(f"Good AUC ({m['AUC']:.4f}), reasonable discriminative ability")
    else:
        obs_parts.append(f"Moderate AUC ({m['AUC']:.4f}), limited discriminative ability")

    # MCC assessment
    if m['MCC'] >= 0.7:
        obs_parts.append(f"Strong MCC ({m['MCC']:.4f}) indicates reliable predictions")
    elif m['MCC'] >= 0.4:
        obs_parts.append(f"Moderate MCC ({m['MCC']:.4f}) indicates acceptable performance")
    else:
        obs_parts.append(f"Low MCC ({m['MCC']:.4f}) suggests room for improvement")

    observation = ". ".join(obs_parts) + "."
    observations[name] = observation
    print(f"\n📌 {name}:")
    print(f"   {observation}")

print(f"\n\n🏆 Overall Winner: {winner}")
print(f"   Best F1 Score: {winner_f1:.4f}")

---
## Step 10: Save Models & Scaler for Streamlit App

In [ ]:
# Create model directory
os.makedirs('model', exist_ok=True)

# Save the scaler
joblib.dump(scaler, 'model/scaler.pkl')
print("✅ Saved: model/scaler.pkl")

# Save all trained models
model_filenames = {
    'Logistic Regression': 'model/logistic_regression.pkl',
    'Decision Tree': 'model/decision_tree.pkl',
    'KNN': 'model/knn.pkl',
    'Naive Bayes': 'model/naive_bayes.pkl',
    'Random Forest': 'model/random_forest.pkl'
}

for name, filepath in model_filenames.items():
    joblib.dump(trained_models[name], filepath)
    print(f"✅ Saved: {filepath}")

# Save the feature names for reference
joblib.dump(list(X.columns), 'model/feature_names.pkl')
print("✅ Saved: model/feature_names.pkl")

print(f"\n📂 All model files saved in 'model/' directory!")

---
## Step 11: Generate the Streamlit App (`app.py`)

This cell writes the complete Streamlit application file.

In [ ]:
%%writefile app.py
# ============================================================
# Heart Disease Classification — Streamlit App
# ML Assignment 2 | M.Tech AIML/DSE | BITS Pilani WILP
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report,
    RocCurveDisplay
)
from sklearn.preprocessing import StandardScaler

# ---- Page Configuration ----
st.set_page_config(
    page_title="Heart Disease Classifier",
    page_icon="🫀",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ---- Custom CSS for Premium Look ----
st.markdown("""
<style>
    .main-header {
        font-size: 2.5rem;
        font-weight: 700;
        color: #1a1a2e;
        text-align: center;
        margin-bottom: 0.5rem;
    }
    .sub-header {
        font-size: 1.1rem;
        color: #6c757d;
        text-align: center;
        margin-bottom: 2rem;
    }
    .metric-card {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 1.2rem;
        border-radius: 12px;
        color: white;
        text-align: center;
        box-shadow: 0 4px 15px rgba(0,0,0,0.1);
    }
    .stMetric {
        background-color: #f8f9fa;
        padding: 10px;
        border-radius: 10px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.05);
    }
</style>
""", unsafe_allow_html=True)

# ---- Title ----
st.markdown('<p class="main-header">🫀 Heart Disease Classification</p>', unsafe_allow_html=True)
st.markdown('<p class="sub-header">ML Assignment 2 — M.Tech (AIML/DSE) | BITS Pilani WILP</p>', unsafe_allow_html=True)
st.markdown('---')

# ---- Model Loading ----
MODEL_DIR = 'model'

model_files = {
    'Logistic Regression': 'logistic_regression.pkl',
    'Decision Tree': 'decision_tree.pkl',
    'KNN': 'knn.pkl',
    'Naive Bayes': 'naive_bayes.pkl',
    'Random Forest': 'random_forest.pkl'
}

@st.cache_resource
def load_models():
    """Load all saved models and scaler."""
    loaded_models = {}
    for name, filename in model_files.items():
        path = os.path.join(MODEL_DIR, filename)
        if os.path.exists(path):
            loaded_models[name] = joblib.load(path)
    scaler = joblib.load(os.path.join(MODEL_DIR, 'scaler.pkl'))
    feature_names = joblib.load(os.path.join(MODEL_DIR, 'feature_names.pkl'))
    return loaded_models, scaler, feature_names

try:
    loaded_models, scaler, feature_names = load_models()
    st.sidebar.success(f"✅ {len(loaded_models)} models loaded successfully!")
except Exception as e:
    st.error(f"❌ Error loading models: {e}")
    st.stop()

# ---- Sidebar ----
st.sidebar.header('⚙️ Configuration')

# Model selection dropdown
selected_model_name = st.sidebar.selectbox(
    '🤖 Select ML Model',
    options=list(loaded_models.keys()),
    index=0,
    help='Choose a classification model to evaluate on the uploaded test data.'
)

compare_all = st.sidebar.checkbox('📊 Compare All Models', value=True,
                                   help='Show comparison table of all models.')

st.sidebar.markdown('---')
st.sidebar.markdown('### 📋 About')
st.sidebar.info(
    '**Dataset:** Heart Disease (UCI/Kaggle)\n\n'
    '**Features:** 13 clinical features\n\n'
    '**Target:** Heart Disease (Yes/No)\n\n'
    '**Models:** 5 ML classifiers'
)

# ---- File Upload ----
st.header('📂 Upload Test Data')
uploaded_file = st.file_uploader(
    'Upload your test data CSV file (must include a `target` column)',
    type=['csv'],
    help='Upload the test_data.csv file. It should contain the same features as the training data plus a target column.'
)

if uploaded_file is not None:
    # Read uploaded file
    test_data = pd.read_csv(uploaded_file)
    st.success(f'✅ File uploaded: {test_data.shape[0]} rows × {test_data.shape[1]} columns')

    # Show preview
    with st.expander('🔍 Preview Uploaded Data', expanded=False):
        st.dataframe(test_data.head(10), use_container_width=True)

    # Validate columns
    if 'target' not in test_data.columns:
        st.error('❌ The uploaded CSV must contain a `target` column!')
        st.stop()

    missing_features = [f for f in feature_names if f not in test_data.columns]
    if missing_features:
        st.error(f'❌ Missing features in uploaded data: {missing_features}')
        st.stop()

    # Separate features and target
    X_uploaded = test_data[feature_names]
    y_uploaded = test_data['target']

    # Scale features
    X_uploaded_scaled = scaler.transform(X_uploaded)

    st.markdown('---')

    # ---- Evaluate Selected Model ----
    st.header(f'🎯 Results: {selected_model_name}')

    selected_model = loaded_models[selected_model_name]
    y_pred = selected_model.predict(X_uploaded_scaled)

    if hasattr(selected_model, 'predict_proba'):
        y_prob = selected_model.predict_proba(X_uploaded_scaled)[:, 1]
        auc_val = roc_auc_score(y_uploaded, y_prob)
    else:
        auc_val = roc_auc_score(y_uploaded, y_pred)

    acc = accuracy_score(y_uploaded, y_pred)
    prec = precision_score(y_uploaded, y_pred, zero_division=0)
    rec = recall_score(y_uploaded, y_pred, zero_division=0)
    f1 = f1_score(y_uploaded, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_uploaded, y_pred)

    # Display metrics in columns
    col1, col2, col3, col4, col5, col6 = st.columns(6)
    col1.metric('Accuracy', f'{acc:.4f}')
    col2.metric('AUC Score', f'{auc_val:.4f}')
    col3.metric('Precision', f'{prec:.4f}')
    col4.metric('Recall', f'{rec:.4f}')
    col5.metric('F1 Score', f'{f1:.4f}')
    col6.metric('MCC', f'{mcc:.4f}')

    st.markdown('---')

    # ---- Confusion Matrix & Classification Report ----
    col_left, col_right = st.columns(2)

    with col_left:
        st.subheader('📉 Confusion Matrix')
        cm = confusion_matrix(y_uploaded, y_pred)
        fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
                    xticklabels=['No Disease', 'Disease'],
                    yticklabels=['No Disease', 'Disease'],
                    cbar=True, linewidths=1, linecolor='black',
                    annot_kws={'size': 16})
        ax_cm.set_xlabel('Predicted', fontsize=12)
        ax_cm.set_ylabel('Actual', fontsize=12)
        ax_cm.set_title(f'Confusion Matrix — {selected_model_name}', fontsize=14, fontweight='bold')
        st.pyplot(fig_cm)
        plt.close(fig_cm)

    with col_right:
        st.subheader('📝 Classification Report')
        report = classification_report(
            y_uploaded, y_pred,
            target_names=['No Disease', 'Disease'],
            output_dict=True
        )
        report_df = pd.DataFrame(report).T
        st.dataframe(report_df.style.format('{:.4f}'), use_container_width=True)

    st.markdown('---')

    # ---- ROC Curve ----
    if hasattr(selected_model, 'predict_proba'):
        st.subheader('📈 ROC Curve')
        fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
        RocCurveDisplay.from_estimator(
            selected_model, X_uploaded_scaled, y_uploaded,
            name=selected_model_name, ax=ax_roc, color='#3498db', linewidth=2
        )
        ax_roc.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
        ax_roc.set_title(f'ROC Curve — {selected_model_name}', fontsize=14, fontweight='bold')
        ax_roc.legend(loc='lower right')
        ax_roc.grid(alpha=0.3)
        st.pyplot(fig_roc)
        plt.close(fig_roc)

    st.markdown('---')

    # ---- Compare All Models ----
    if compare_all:
        st.header('📊 All Models — Comparison Table')

        all_results = {}
        for name, model in loaded_models.items():
            y_p = model.predict(X_uploaded_scaled)
            if hasattr(model, 'predict_proba'):
                y_pr = model.predict_proba(X_uploaded_scaled)[:, 1]
                auc_v = roc_auc_score(y_uploaded, y_pr)
            else:
                auc_v = roc_auc_score(y_uploaded, y_p)

            all_results[name] = {
                'Accuracy': accuracy_score(y_uploaded, y_p),
                'AUC': auc_v,
                'Precision': precision_score(y_uploaded, y_p, zero_division=0),
                'Recall': recall_score(y_uploaded, y_p, zero_division=0),
                'F1': f1_score(y_uploaded, y_p, zero_division=0),
                'MCC': matthews_corrcoef(y_uploaded, y_p)
            }

        comp_df = pd.DataFrame(all_results).T.round(4)
        comp_df.index.name = 'ML Model'

        # Highlight best values
        def highlight_max(s):
            is_max = s == s.max()
            return ['background-color: #2ecc71; color: white; font-weight: bold' if v else '' for v in is_max]

        st.dataframe(
            comp_df.style.apply(highlight_max, axis=0).format('{:.4f}'),
            use_container_width=True
        )

        # Winner
        best_model = comp_df['F1'].idxmax()
        st.success(f'🏆 **Best Model (by F1 Score): {best_model}** — F1: {comp_df.loc[best_model, "F1"]:.4f}')

        # Bar chart comparison
        st.subheader('📊 Visual Comparison')
        fig_comp, ax_comp = plt.subplots(figsize=(12, 5))
        comp_df.plot(kind='bar', ax=ax_comp, edgecolor='black', alpha=0.85, width=0.8)
        ax_comp.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
        ax_comp.set_xlabel('Model', fontsize=12)
        ax_comp.set_ylabel('Score', fontsize=12)
        ax_comp.set_ylim(0, 1.1)
        ax_comp.legend(loc='upper right', ncol=3)
        ax_comp.grid(axis='y', alpha=0.3)
        plt.xticks(rotation=15)
        plt.tight_layout()
        st.pyplot(fig_comp)
        plt.close(fig_comp)

else:
    st.info('👆 Please upload a test data CSV file to begin evaluation.')
    st.markdown(
        '**Expected CSV format:** The file should contain the same feature columns as the training data '
        '(`age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`, `oldpeak`, '
        '`slope`, `ca`, `thal`) plus a `target` column.'
    )

# ---- Footer ----
st.markdown('---')
st.markdown(
    '<div style="text-align:center; color:#888; font-size:0.9rem;">'
    '🫀 Heart Disease Classification App | ML Assignment 2 | BITS Pilani WILP'
    '</div>',
    unsafe_allow_html=True
)

---
## Step 12: Generate `requirements.txt`

In [ ]:
%%writefile requirements.txt
streamlit
scikit-learn
numpy
pandas
matplotlib
seaborn
joblib

---
## Step 13: Generate `README.md`

This cell auto-generates a README based on the actual results from the experiments above.

In [ ]:
# Auto-generate README.md with actual results
readme_content = f"""# 🫀 Heart Disease Classification — ML Assignment 2

**Course:** Machine Learning | M.Tech (AIML/DSE) | BITS Pilani WILP

---

## a. Problem Statement

Predict the presence or absence of heart disease in a patient based on 13 clinical and diagnostic features using multiple supervised classification models. This is a **binary classification** problem where the target variable indicates:
- `1` → Heart Disease Present
- `0` → No Heart Disease

---

## b. Dataset Description

| Property | Value |
|----------|-------|
| **Name** | Heart Disease Dataset |
| **Source** | [Kaggle — Heart Disease Dataset](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset) |
| **Original Source** | UCI Machine Learning Repository |
| **Instances** | {len(df)} (after removing duplicates) |
| **Features** | 13 predictive features + 1 target |
| **Task** | Binary Classification |

### Features

| # | Feature | Type | Description |
|---|---------|------|-------------|
| 1 | `age` | Numeric | Age in years |
| 2 | `sex` | Categorical | 1=Male, 0=Female |
| 3 | `cp` | Categorical | Chest pain type (0-3) |
| 4 | `trestbps` | Numeric | Resting blood pressure (mm Hg) |
| 5 | `chol` | Numeric | Serum cholesterol (mg/dl) |
| 6 | `fbs` | Categorical | Fasting blood sugar > 120 mg/dl |
| 7 | `restecg` | Categorical | Resting ECG results |
| 8 | `thalach` | Numeric | Max heart rate achieved |
| 9 | `exang` | Categorical | Exercise-induced angina |
| 10 | `oldpeak` | Numeric | ST depression (exercise vs rest) |
| 11 | `slope` | Categorical | Slope of peak exercise ST segment |
| 12 | `ca` | Numeric | Major vessels colored by fluoroscopy (0-3) |
| 13 | `thal` | Categorical | Thalassemia type |

---

## c. GitHub Repository Link

> **TODO:** Replace with your actual GitHub repository link after pushing the code.
>
> `https://github.com/<your-username>/<repo-name>`

---

## d. Models Used & Comparison Table

| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
|---------------|----------|-----|-----------|--------|----|----|\n"""

for name in model_names:
    m = results[name]
    readme_content += f"| {name} | {m['Accuracy']:.4f} | {m['AUC']:.4f} | {m['Precision']:.4f} | {m['Recall']:.4f} | {m['F1']:.4f} | {m['MCC']:.4f} |\n"

readme_content += f"""
---

### Model Observations

| ML Model Name | Observation about model performance |
|---------------|------------------------------------|
"""

for name in model_names:
    readme_content += f"| {name} | {observations[name]} |\n"

readme_content += f"| **Overall Winner** | **{winner}** — Best F1 Score ({winner_f1:.4f}), providing the best balance of precision and recall on the Heart Disease dataset. |\n"

readme_content += f"""
---

## Streamlit App

> **TODO:** Replace with your live Streamlit app link after deployment.
>
> `https://<your-app-name>.streamlit.app`

### App Features
- ✅ CSV file upload for test data
- ✅ Model selection dropdown (5 models)
- ✅ Display of all 6 evaluation metrics
- ✅ Confusion matrix visualization
- ✅ Classification report
- ✅ ROC Curve
- ✅ All-models comparison table with visual bar chart

---

## Repository Structure

```
project-folder/
│── app.py                    # Streamlit web application
│── requirements.txt          # Python dependencies
│── README.md                 # This file
│── test_data.csv             # Test data for evaluation
│── model/
│   ├── scaler.pkl            # Fitted StandardScaler
│   ├── feature_names.pkl     # Feature column names
│   ├── logistic_regression.pkl
│   ├── decision_tree.pkl
│   ├── knn.pkl
│   ├── naive_bayes.pkl
│   └── random_forest.pkl
│── ML_Assignment_2.ipynb     # Colab notebook with full analysis
```

---

## How to Run Locally

```bash
pip install -r requirements.txt
streamlit run app.py
```

---

## Deployment on Streamlit Community Cloud

1. Push all files to GitHub
2. Go to [streamlit.io/cloud](https://streamlit.io/cloud)
3. Sign in with GitHub
4. Click "New App" → Select your repo → Choose `app.py` → Deploy
"""

with open('README.md', 'w') as f:
    f.write(readme_content)

print("✅ README.md generated successfully!")
print("\n📋 Preview (first 30 lines):")
print("\n".join(readme_content.split('\n')[:30]))

---
## Step 14: Download All Files

Run this cell to download all files needed for the GitHub repository.

In [ ]:
# Create a zip file with all project files for easy download
import shutil

# Define the project folder
project_name = 'heart_disease_ml_project'

# Clean up if exists
if os.path.exists(project_name):
    shutil.rmtree(project_name)

os.makedirs(f'{project_name}/model', exist_ok=True)

# Copy files
shutil.copy('app.py', f'{project_name}/app.py')
shutil.copy('requirements.txt', f'{project_name}/requirements.txt')
shutil.copy('README.md', f'{project_name}/README.md')
shutil.copy('test_data.csv', f'{project_name}/test_data.csv')

# Copy model files
for f in os.listdir('model'):
    shutil.copy(f'model/{f}', f'{project_name}/model/{f}')

# Create zip
shutil.make_archive(project_name, 'zip', '.', project_name)

print(f"✅ Project zip created: {project_name}.zip")
print(f"\n📂 Contents:")
for root, dirs, files_list in os.walk(project_name):
    level = root.replace(project_name, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files_list:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        print(f'{subindent}{file} ({size:,} bytes)')

In [ ]:
# Download the zip file
from google.colab import files

print("📥 Downloading project zip...")
files.download(f'{project_name}.zip')
print("\n✅ Download started! Check your browser's download folder.")

In [ ]:
# Also download individual important files (optional)
print("📥 You can also download individual files:")
print("   Uncomment the lines below to download specific files.")

# files.download('app.py')
# files.download('requirements.txt')
# files.download('README.md')
# files.download('test_data.csv')

---
## Step 15: Test Streamlit App Locally (Optional)

Run this cell to test the Streamlit app directly in Colab using localtunnel.

In [ ]:
# Uncomment and run to test Streamlit in Colab
# !pip install -q streamlit
# !npm install -g localtunnel

# # Run Streamlit in background and tunnel
# import subprocess
# process = subprocess.Popen(
#     ['streamlit', 'run', 'app.py', '--server.port', '8501'],
#     stdout=subprocess.PIPE, stderr=subprocess.PIPE
# )
# !npx localtunnel --port 8501

---
## ✅ Assignment Checklist

| # | Requirement | Status |
|---|-------------|--------|
| 1 | Dataset: ≥12 features, ≥500 instances | ✅ Heart Disease (13 features, 1025 instances) |
| 2 | Logistic Regression | ✅ Implemented & Evaluated |
| 3 | Decision Tree Classifier | ✅ Implemented & Evaluated |
| 4 | K-Nearest Neighbor Classifier | ✅ Implemented & Evaluated |
| 5 | Naive Bayes (Gaussian) | ✅ Implemented & Evaluated |
| 6 | Random Forest (Ensemble) | ✅ Implemented & Evaluated |
| 7 | Accuracy metric | ✅ Calculated |
| 8 | AUC Score | ✅ Calculated |
| 9 | Precision | ✅ Calculated |
| 10 | Recall | ✅ Calculated |
| 11 | F1 Score | ✅ Calculated |
| 12 | MCC Score | ✅ Calculated |
| 13 | Comparison Table | ✅ Generated |
| 14 | Model Observations | ✅ Auto-generated |
| 15 | app.py (Streamlit) | ✅ Generated |
| 16 | requirements.txt | ✅ Generated |
| 17 | README.md | ✅ Auto-generated with results |
| 18 | test_data.csv | ✅ Saved |
| 19 | Model files (.pkl) | ✅ Saved in model/ |
| 20 | Streamlit: CSV upload | ✅ Implemented |
| 21 | Streamlit: Model dropdown | ✅ Implemented |
| 22 | Streamlit: Evaluation metrics | ✅ Implemented |
| 23 | Streamlit: Confusion matrix | ✅ Implemented |

### 📝 Remaining Steps (Manual):
1. **Download the zip** → Extract → Push to GitHub
2. **Deploy on Streamlit Cloud** (streamlit.io/cloud → New App → Select repo)
3. **Update README.md** with actual GitHub repo link and Streamlit app link
4. **Take screenshot** on BITS Virtual Lab
5. **Create submission PDF** with all required links and README content